## Title: Luong Attention (Multiplicative Attention)

### Objectives
* Build an RNN-based Encoder–Decoder model integrated with the Luong (Multiplicative) Attention mechanism for English–French machine translation using the PyTorch framework.
* Explore how Luong Attention enhances translation accuracy by allowing the decoder to dynamically concentrate on the most relevant encoder hidden states through multiplicative alignment scoring during each decoding step.

#### Theory

The Luong Attention model is an improved RNN Encoder–Decoder architecture used for English–French machine translation. The encoder reads the input sentence and generates hidden states for each word. During decoding, the decoder compares its current hidden state with all encoder hidden states using a multiplicative (dot-product) scoring method. The attention scores are normalized with Softmax to obtain attention weights, which help create a context vector containing the most relevant information from the input sentence. This context vector is combined with the decoder output to predict the next word, improving translation accuracy, especially for longer sentences. The model is trained using teacher forcing, optimized with the Adam optimizer, and uses NLL loss for error calculation.

In [21]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
from torch.utils.data import TensorDataset

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [22]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [23]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [24]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [25]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [26]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [27]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am'] 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes bougon', 'you re grumpy']


15

In [28]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)
                weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )
# Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

In [30]:
class LuongDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(LuongDecoderRNN, self).__init__()

        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)

        self.dropout = nn.Dropout(dropout_p)

        self.rnn = nn.RNN(
            hidden_size,
            hidden_size,
            batch_first=True
        )

        self.attention = LuongDotAttention(hidden_size)

        self.concat = nn.Linear(
            hidden_size * 2,
            hidden_size
        )

        self.out = nn.Linear(
            hidden_size,
            output_size
        )
        
    def forward(self,
                encoder_outputs,
                encoder_hidden,
                target_tensor=None):

        batch_size = encoder_outputs.size(0)

        decoder_input = torch.full(
            (batch_size, 1),
            SOS_token,
            dtype=torch.long,
            device=device
        )

        decoder_hidden = encoder_hidden

        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):

            decoder_output, decoder_hidden, attn = self.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self,
                     input,
                     hidden,
                     encoder_outputs):

        embedded = self.dropout(
            self.embedding(input)
        )
# Decoder RNN first
        rnn_output, hidden = self.rnn(
            embedded,
            hidden
        )

        # Prepare query
        query = rnn_output

        # Luong attention
        context, attn_weights = self.attention(
            query,
            encoder_outputs
        )

        # Concatenate context and decoder output
        concat_input = torch.cat(
            (rnn_output, context),
            dim=2
        )

        concat_output = torch.tanh(
            self.concat(concat_input)
        )

        output = self.out(concat_output)

        return output, hidden, attn_weights

In [31]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [32]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [33]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [34]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [35]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [36]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [37]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [38]:
from torch.utils.data import TensorDataset
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = LuongDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 8s (- 5m 46s) (5 2%) 1.8208
0m 17s (- 5m 38s) (10 5%) 1.2044
0m 27s (- 5m 39s) (15 7%) 0.9533
0m 37s (- 5m 35s) (20 10%) 0.7847
0m 47s (- 5m 32s) (25 12%) 0.6686
0m 56s (- 5m 21s) (30 15%) 0.5598
1m 5s (- 5m 10s) (35 17%) 0.4682
1m 15s (- 5m 2s) (40 20%) 0.3756
1m 24s (- 4m 51s) (45 22%) 0.2936
1m 33s (- 4m 41s) (50 25%) 0.2413
1m 43s (- 4m 33s) (55 27%) 0.1958
1m 52s (- 4m 23s) (60 30%) 0.1726
2m 1s (- 4m 11s) (65 32%) 0.1508
2m 10s (- 4m 2s) (70 35%) 0.1343
2m 20s (- 3m 53s) (75 37%) 0.1248
2m 29s (- 3m 43s) (80 40%) 0.1177
2m 39s (- 3m 35s) (85 42%) 0.1123
2m 47s (- 3m 24s) (90 45%) 0.1030
2m 56s (- 3m 15s) (95 47%) 0.0977
3m 5s (- 3m 5s) (100 50%) 0.0939
3m 14s (- 2m 56s) (105 52%) 0.0878
3m 23s (- 2m 46s) (110 55%) 0.0854
3m 33s (- 2m 37s) (115 57%) 0.0846
3m 42s (- 2m 28s) (120 60%) 0.0842
3m 51s (- 2m 19s) (125 62%) 0.0789
4m 0s (- 2m 9s) (130 65%) 0.07

In [39]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis meticuleuse
= i m finicky
< i m first <EOS>

> il est apres moi
= he s after me
< he s after me <EOS>

> je suis ton partenaire
= i m your partner
< i m your partner <EOS>

> tu es fort grossier
= you re very rude
< you re very rude <EOS>

> je suis timide
= i m shy
< i m shy <EOS>

> nous sommes trop vieux
= we re too old
< we re too old <EOS>

> je viens d amerique
= i m from america
< i m from america <EOS>

> je suis terrifiee
= i m terrified
< i m hungry <EOS>

> je suis fauche
= i m broke
< i m all ears <EOS>

> nous sommes tous amis
= we re all friends
< we re all friends <EOS>



### Discussion
The implemented RNN Encoder–Decoder model with Luong Attention performs English–French machine translation by allowing the decoder to attend to important parts of the input sentence during each decoding step. The encoder generates hidden states for the input sequence, while the attention mechanism helps the decoder select the most relevant information before predicting the next word. The model is trained using teacher forcing, the Adam optimizer, and NLL loss. A gradual decrease in training loss indicates that the model is learning effectively and improving its translation accuracy.

### Conclusion
The Luong Attention mechanism enhances the Encoder–Decoder model by enabling the decoder to focus on relevant information from the input sentence instead of relying only on the final encoder state. This results in better translation quality, especially for longer sentences. Overall, the implementation provides a practical understanding of attention-based neural machine translation using PyTorch.